# Problem Set 6: Text Analysis of DOJ Press Releases, Part 1

- For background:
    - DOJ is the federal law enforcement agency responsible for federal prosecutions; this contrasts with the local prosecutions in the Cook County dataset we analyzed earlier. Here's a short explainer on which crimes get prosecuted federally versus locally: https://www.criminaldefenselawyer.com/resources/criminal-defense/federal-crime/state-vs-federal-crimes.htm#:~:text=Federal%20criminal%20prosecutions%20are%20handled,of%20state%20and%20local%20law. 
    - Here's the Kaggle that contains the data: https://www.kaggle.com/jbencina/department-of-justice-20092018-press-releases 

0.0 Import packages

In [10]:
## helpful packages
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import random
import re
import string

## Install relevant packages if you haven't already
## ! conda activate qss20
## ! conda install nltk spacy vaderSentiment gensim scikit-learn "scipy<1.13"

## nltk imports
import nltk
nltk.download('averaged_perceptron_tagger_eng')
### uncomment and run these lines if you haven't downloaded relevant nltk add-ons yet
#nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
from nltk import pos_tag
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords

## spacy imports
import spacy
### uncomment and run the below line if you haven't loaded the en_core_web_sm library yet
### ! python -m spacy download en_core_web_sm
import en_core_web_sm
nlp = en_core_web_sm.load()

## vectorizer
from sklearn.feature_extraction.text import CountVectorizer

## sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## LDA
from gensim import corpora
import gensim

## repeated printouts and wide-format text
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
pd.set_option('display.max_colwidth', None)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/vivianaperez/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vivianaperez/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

0.1 Load and clean text data

In [59]:
## first, unzip the file pset6_inputdata.zip 
## then, run this code to load the unzipped json file and convert to a dataframe
## (may need to change the pathname depending on where you store stuff)
## and convert some of the attributes from lists to values

doj = pd.read_json("combined.json", lines = True)

## due to json, topics are in a list so remove them and concatenate with ;
doj['topics_clean'] = ["; ".join(topic) 
                      if len(topic) > 0 else "No topic" 
                      for topic in doj.topics]

## similarly with components
doj['components_clean'] = ["; ".join(comp) 
                           if len(comp) > 0 else "No component" 
                           for comp in doj.components]

## drop older columns from data
doj = doj[['id', 'title', 'contents', 'date', 'topics_clean', 
           'components_clean']].copy()

doj.head()


id  \
0     None   
1  12-919    
2  11-1002   
3   10-015   
4   18-898   

                                                                                                          title  \
0                                                                  Convicted Bomb Plotter Sentenced to 30 Years   
1                              $1 Million in Restitution Payments Announced to Preserve North Carolina Wetlands   
2                 $1 Million Settlement Reached for Natural Resource Damages at Superfund Site in Massachusetts   
3                                          10 Las Vegas Men Indicted \r\nfor Falsifying Vehicle Emissions Tests   
4  $100 Million Settlement Will Speed Cleanup Work at Centredale Manor Superfund Site in North Providence, R.I.   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

## 1. Tagging and sentiment scoring (17 points)

Focus on the following press release: `id` == "17-1204" about this pharmaceutical kickback prosecution: https://www.forbes.com/sites/michelatindera/2017/11/16/fentanyl-billionaire-john-kapoor-to-plead-not-guilty-in-opioid-kickback-case/?sh=21b8574d6c6c 

The `contents` column is the one we're treating as a document. You may need to to convert it from a pandas series to a single string.

We'll call the raw string of this press release `pharma`

In [3]:
## your code to subset to one press release and take the string

# subsets to id 17-1204
pharma_df = doj[doj['id'] == "17-1204"]

# makes into a single string
pharma = pharma_df['contents'].iloc[0]

# check
print(pharma[:500])

The founder and majority owner of Insys Therapeutics Inc., was arrested today and charged with leading a nationwide conspiracy to profit by using bribes and fraud to cause the illegal distribution of a Fentanyl spray intended for cancer patients experiencing breakthrough pain. "More than 20,000 Americans died of synthetic opioid overdoses last year, and millions are addicted to opioids. And yet some medical professionals would rather take advantage of the addicts than try to help them," said Att


### 1.1 part of speech tagging (3 points)

A. Preprocess the `pharma` press release to remove all punctuation / digits (you can use `.isalpha()` to subset)

B. With the preprocessed press release from part A, use the part of speech tagger within nltk to tag all the words in that one press release with their part of speech. 

C. Using the output from B, extract the adjectives and sort those adjectives from most occurrences to fewest occurrences. Print a dataframe with the 5 most frequent adjectives and their counts in the `pharma` release. See here for a list of the names of adjectives within nltk: https://pythonprogramming.net/natural-language-toolkit-nltk-part-speech-tagging/

**Resources**:

- Documentation for `.isalpha()`: https://www.w3schools.com/python/ref_string_isalpha.asp

In [ ]:
# part a
# split into tokens based on spaces
tokens = pharma.split()

# keeps only the alphabetic tokens and converts to all lowercase
tokens_alpha = [word.lower() for word in tokens if word.isalpha()]

# check
print(tokens_alpha[:20])

['the', 'founder', 'and', 'majority', 'owner', 'of', 'insys', 'therapeutics', 'was', 'arrested', 'today', 'and', 'charged', 'with', 'leading', 'a', 'nationwide', 'conspiracy', 'to', 'profit']


In [ ]:
# part b
# tags the tokens by part of speach 
tagged_tokens = pos_tag(tokens)

#(part c)
# extracts adjectives
adjectives = [word for word, tag in tagged_tokens 
              if tag in ['JJ', 'JJR', 'JJS']]
# makes into a dataframe
adj_df = pd.DataFrame(adjectives, columns=['adjective'])
# counts occurrences of adjectives and renames col
adj_counts = adj_df.value_counts().reset_index()

adj_counts.head(5)



,adjective,count
0,former,8
1,opioid,5
2,nationwide,4
3,addictive,3
4,other,3


## 1.2 named entity recognition (4 points)

A. Using the original `pharma` press release (so the one before stripping punctuation/digits), use spaCy to extract all named entities from the press release.

B. Print the unique named entities with the tag: `LAW`

In [41]:
# part a
doc = nlp(pharma)  

# getting named entities as tuples
entities = [(ent.text, ent.label_) for ent in doc.ents]

# check
entities[:20]

[('Insys Therapeutics Inc.', 'ORG'),
 ('today', 'DATE'),
 ('Fentanyl', 'PERSON'),
 ('More than 20,000', 'CARDINAL'),
 ('Americans', 'NORP'),
 ('last year', 'DATE'),
 ('millions', 'CARDINAL'),
 ('Jeff Sessions', 'PERSON'),
 ('This Justice Department', 'ORG'),
 ('Trump', 'PERSON'),
 ('American', 'NORP'),
 ('”John N. Kapoor', 'PERSON'),
 ('74', 'DATE'),
 ('Phoenix', 'GPE'),
 ('Ariz.', 'GPE'),
 ('the Board of Directors', 'ORG'),
 ('Insys', 'FAC'),
 ('this morning', 'TIME'),
 ('Arizona', 'GPE'),
 ('RICO', 'LAW')]

In [42]:
# part b
law_entities = list(set([ent.text 
                         for ent in doc.ents 
                         if ent.label_ == "LAW"]))

print(law_entities)

['the Controlled Substances Act', 'RICO']


C. Use Google to summarize in one sentence what the `RICO` named entity means and why this might apply to a pharmaceutical kickbacks case (and not just a mafia case...) 

RICO  is a law that allows prosecutors to charge people who particiapte in an organized pattern of criminal activity, which can apply to pharmacutical kickbacks if executives worked together in a coordinated scheme.

In [ ]:
## your code here 

D. You want to extract the possible sentence lengths the CEO is facing; pull out the named entities with (1) the label `DATE` and (2) that contain the word year or years (hint: you may want to use the `re` module for that second part). Print these named entities.

In [45]:
# extract sentence length
sentence_lengths = []

for ent in doc.ents:
    if ent.label_ == "DATE":
        if re.search(r'\byears?\b', ent.text.lower()):
            sentence_lengths.append(ent.text)

print(sentence_lengths)

['last year', '20 years', 'three years', 'five years', 'three years']


E. Pull and print the original parts of the press releases where those year lengths are mentioned (e.g., the sentences or rough region of the press release). Describe in your own words (1 sentence) what length of sentence (prison) and probation (supervised release) the CEO may be facing if convicted after this indictment (if there are multiple lengths mentioned describe the maximum). 

**Hint**: you may want to use re.search or re.findall 

- For part E, you can use `re.search` and `re.findall`, or anything that works 😳.

In [ ]:
# splits into sentences by period
sentences = re.split(r'(?<=[.!?])\s+', pharma)

relevant_sentences = []

# goes through all sentences and checks if it has 'year' or 'years'
for sentence in sentences:
    if re.search(r'\byears?\b', sentence.lower()):
        relevant_sentences.append(sentence)

# print them
for s in relevant_sentences:
    print(s)

# if convidencted, the ceo could face up to 20 years with three years 
# of supervised release for the RICO charges and up to five years in prison
# with three years of supervised realise of the Anti-Kickback charge.

"More than 20,000 Americans died of synthetic opioid overdoses last year, and millions are addicted to opioids.
Neves, Special Agent in Charge of the VA OIG Northeast Field Office.The charges of conspiracy to commit RICO and conspiracy to commit mail and wire fraud each provide for a sentence of no greater than 20 years in prison, three years of supervised release and a fine of $250,000, or twice the amount of pecuniary gain or loss.
The charges of conspiracy to violate the Anti-Kickback Law provide for a sentence of no greater than five years in prison, three years of supervised release and a $25,000 fine.


## 1.3 sentiment analysis  (10 points)

A. Subset the press releases to those labeled with one of three topics via `topics_clean`: Civil Rights, Hate Crimes, and Project Safe Childhood. We'll call this `doj_subset` going forward and it should have 717 rows.



In [ ]:
## your code here for subsetting
# subset to the three topics
doj_subset = doj[doj['topics_clean'].str.contains(
    r'Civil Rights|Hate Crimes|Project Safe Childhood',
    regex=True
)].copy()

# check
doj_subset.shape # cant get 717 :(


(856, 6)

(856, 6)

B. Write a function that takes one press release string as an input and:

- Removes named entities from each press release string (**Hint**: you may want to use `re.sub` with an or condition)
- Scores the sentiment of the entire press release using the `SentimentIntensityAnalyzer` and `polarity_scores`
- Returns the length-four (negative, positive, neutral, compound) sentiment dictionary (any order is fine)

Apply that function to each of the press releases in `doj_subset`. 

**Hints**: 

- A function + list comprehension to execute will takes about 30 seconds on a respectable local machine and about 2 mins on jhub; if it's taking a very long time, you may want to check your code for inefficiencies. If you can't fix those, for partial credit on this part/full credit on remainder, you can take a small random sample of the 717


In [ ]:
# defining function
def sentiment_no_entities(text):
    # processes text
    doc = nlp(text)
    
    # creates a regex pattern matching the  named entities
    entities = [re.escape(ent.text) for ent in doc.ents]
    if entities:
        pattern = r'\b(?:' + '|'.join(entities) + r')\b'
        text_no_entities = re.sub(pattern, '', text)
    else:
        text_no_entities = text
    
    # sentiment
    sentiment = analyzer.polarity_scores(text_no_entities)
    return sentiment


In [ ]:
## your code here executing the function
sentiment_results = [sentiment_no_entities(text) 
                     for text in doj_subset['contents']]

sentiment_df = pd.DataFrame(sentiment_results)

# check
sentiment_df.head()

,neg,neu,pos,compound
0,0.141,0.758,0.101,-0.9920
1,0.199,0.752,0.049,-0.9931
2,0.132,0.800,0.068,-0.9325
3,0.092,0.832,0.076,-0.7579
4,0.125,0.792,0.084,-0.9037


C. Add the four sentiment scores to the `doj_subset` dataframe to create a dataframe: `doj_subset_wscore`. Sort from highest neg to lowest neg score and print the top `id`, `contents`, and `neg` columns of the two most neg press releases. 

Notes:

- Don't worry if your sentiment score differs slightly from our output on GitHub; differences in preprocessing can lead to diff scores

In [ ]:
# resets index
doj_subset_wscore = doj_subset.reset_index(drop=True).copy()

doj_subset_wscore[['neg', 'neu', 'pos', 'compound']] = sentiment_df

# sorts by sentiment
doj_sorted_neg = doj_subset_wscore.sort_values(by='neg', ascending=False)
doj_sorted_neg[['id', 'contents', 'neg']].head(2) # these two are the most negative

id  \
0   16-659   
1  17-1235   
2  15-1522   
3   16-213   
4   16-381   

                                                                                                                      title  \
0                                                       Additional Defendants Sentenced for Roles in Sex Trafficking Scheme   
1  Additional Former Correctional Officer Pleads Guilty to Beating of Handcuffed and Shackled Inmate at Angola State Prison   
2                                                            Alabama Man Found Guilty of Aggravated Sexual Abuse of a Child   
3                                                        Alabama Man Indicted on Child Pornography and Sex Tourism Charges    
4                                           Alabama Man Indicted for Producing Child Pornography Involving Multiple Victims   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

,id,contents,neg
14,14-248,"The Department of Justice announced that this morning John W. Ng, 58, of Albuquerque, N.M., made his initial appearance in federal court on a criminal complaint charging him with a hate crime offense. This charge is related to anti-Semitic threats Ng made against a Jewish woman who owns and operates the Nosh Jewish Delicatessen and Bakery in Albuquerque. Ng was arrested by the FBI on March 7, 2014, based on a criminal complaint alleging that he interfered with the victim’s federally protected rights by threatening her and interfering with her business because of her religion. According to the criminal complaint, between Jan. 22, 2014, and Feb. 8, 2014, Ng allegedly posted threatening anti-Semitic notes on and in the vicinity of the victim’s business. A criminal complaint merely establishes probable cause, and Ng is presumed innocent unless proven guilty. If convicted on the offense charged in the criminal complaint, Ng faces a maximum statutory penalty of one year in prison. This matter was investigated by the Albuquerque Division of the FBI and is being prosecuted by Assistant U.S. Attorney Mark T. Baker of the U.S. Attorney’s Office for the District of New Mexico and Trial Attorney AeJean Cha of the U.S. Department of Justice’s Civil Rights Division.",0.308
35,13-312,"John Hall, 27, an Aryan Brotherhood member and inmate at the Federal Correctional Institution (FCI) in Seagoville, Texas, was sentenced today by U.S. District Judge Reed O’Connor after pleading guilty to violating the Matthew Shepard and James Byrd Jr. Hate Crimes Prevention Act stemming from his assault of a fellow inmate, whom he believed to be gay, the Department of Justice announced. Hall assaulted his fellow inmate with a dangerous weapon, causing bodily injury to the victim on Dec. 20, 2011. Hall was sentenced to serve 71 months in prison to be served consecutively with the sentence he is currently serving. The assault occurred on Dec. 20, 2011, inside the FCI Seagoville when Hall targeted and attacked the victim, a fellow inmate, because he believed the victim was gay or involved in a sexual relationship with another male inmate. Hall repeatedly punched, kicked and stomped on the victim’s face with his shod feet, a dangerous weapon, while yelling a homophobic slur. The victim lost consciousness during the assault and suffered multiple lacerations to his face. The victim also sustained a fractured eye socket, lost a tooth, fractured other teeth and was treated at a hospital for the injuries he sustained during Hall’s unprovoked attack. Hall pleaded guilty to violating the Matthew Shepard and James Byrd Jr. Hate Crimes Prevention Act on Nov. 8, 2012. “Brutality and violence based on sexual orientation has no place in a civilized society,” said Thomas E. Perez, Assistant Attorney General for the Civil Rights Division. “The Justice Department is committed to using all the tools in our law enforcement arsenal, including the Matthew Shepard and James Byrd Jr. Hate Crimes Prevention Act, to prosecute acts motivated by hate.” “This prosecution sends a clear message that this office, in partnership with attorneys in the department’s Civil Rights Division, will prioritize and aggressively prosecute hate crimes and others civil rights violations in North Texas,” said U.S. Attorney Sarah R. Saldaña of the Northern District of Texas. This case was investigated by the FBI Dallas Division. The case was prosecuted by Assistant U.S. Attorney Errin Martin and Trial Attorney Adriana Vieco of the Civil Rights Division.",0.301


D. With the dataframe from part C, find the mean compound sentiment score for each of the three topics in `topics_clean` using group_by and agg.

E. Add a 1 sentence interpretation of why we might see the variation in scores (remember that compound is a standardized summary where -1 is most negative; +1 is most positive)


In [ ]:

mean_compound = (doj_subset_wscore.groupby('topics_clean')
        .agg(mean_compound=('compound', 'mean'))
        .reset_index()
)

print(mean_compound)

# Variation in scores likely shows differences in 
# nature of cases discussed since topics with things like violent crimes, trafficking,
# or hate crimes have very strong negative language whereas other topics
#like public corruption convictions, or outreach initiatives used
# more positive language.

                                                   topics_clean  mean_compound
0                               Access to Justice; Civil Rights       0.969117
1                                                  Civil Rights      -0.089995
2             Civil Rights; Counterterrorism; National Security      -0.992400
3                                     Civil Rights; Cyber Crime      -0.998900
4                                Civil Rights; Drug Trafficking      -0.997300
5             Civil Rights; Drug Trafficking; Human Trafficking      -0.999200
6                               Civil Rights; Firearms Offenses      -0.997300
7                                     Civil Rights; Hate Crimes      -0.838915
8                               Civil Rights; Human Trafficking      -0.978522
9                                     Civil Rights; Immigration       0.610992
10                 Civil Rights; Indian Country Law and Justice      -0.961200
11                             Civil Rights; Labor &

# 2. Optional extra credit (2 points)

You notice that the pharmaceutical kickbacks press release we analyzed in question 1 was for an indictment, and that in the original data, there's not a clear label for whether a press release outlines an indictment (charging someone with a crime), a conviction (convicting them after that charge either via a settlement or trial), or a sentencing (how many years of prison or supervised release a defendant is sentenced to after their conviction).

You want to see if you can identify pairs of press releases where one press release is from one stage (e.g., indictment) and another is from a different stage (e.g., a sentencing).

You decide that one way to approach is to find the pairwise string similarity between each of the processed press releases in `doj_subset`. There are many ways to do this, so Google for some approaches, focusing on ones that work well for entire documents rather than small strings.

Find the top two pairs (so four press releases total)-- do they seem like different stages of the same crime or just press releases covering similar crimes?